In [0]:
dbutils.widgets.dropdown("subfolder", "products",["products", "categories", "inventory", "clickstream"],"File Source",)

In [0]:
dbutils.widgets.dropdown("catalog", "dev", ["dev", "prod"])

In [0]:
subfolder = dbutils.widgets.get("subfolder")
catalog = dbutils.widgets.get("catalog")
print(f"Selected source: {subfolder} and catalog: {catalog}")


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, BooleanType, LongType, DoubleType)

In [0]:
source_path = f"/Volumes/{catalog}/stepright/landing/{subfolder}/"

print(source_path)

In [0]:
target_table= f"{catalog}.os_stepright.bronze_{subfolder}"
print(target_table)

In [0]:
checkpoint_loc = f"/Volumes/{catalog}/os_stepright/checkpoints/bronze_{subfolder}"
print(checkpoint_loc)

In [0]:
FILE_FORMAT_MAP = {
    "products": "csv",
    "categories": "csv",
    "inventory": "csv",
    "clickstream": "json",
}

####Schema Declaration for all File sources

In [0]:
products_struct_column_schema = StructType(
    [
        StructField("product_id", StringType()),
        StructField("sku", StringType()),
        StructField("product_name", StringType()),
        StructField("brand", StringType()),
        StructField("category_id", StringType()),
        StructField("gender_target", StringType()),
        StructField("size_uk", DoubleType()),
        StructField("colour", StringType()),
        StructField("material", StringType()),
        StructField("cost_price", DoubleType()),
        StructField("retail_price", DoubleType()),
        StructField("is_active", BooleanType()),
        StructField("launch_date", StringType()),
    ]
)

categories_struct_column_schema = StructType(
    [
        StructField("category_id", StringType()),
        StructField("category_name", StringType()),
        StructField("parent_category_id", StringType()),
        StructField("is_active", BooleanType()),
    ]
)

inventory_struct_column_schema = StructType(
    [
        StructField("snapshot_id", StringType()),
        StructField("snapshot_date", StringType()),
        StructField("product_id", StringType()),
        StructField("sku", StringType()),
        StructField("warehouse_id", StringType()),
        StructField("quantity_on_hand", LongType()),
        StructField("quantity_reserved", LongType()),
        StructField("quantity_available", LongType()),
        StructField("reorder_point", LongType()),
        StructField("days_of_supply", LongType()),
    ]
)

clickstream_struct_column_schema = StructType(
    [
        StructField("event_id", StringType()),
        StructField("session_id", StringType()),
        StructField("customer_id", StringType()),
        StructField("event_type", StringType()),
        StructField("event_timestamp", StringType()),
        StructField("product_id", StringType()),
        StructField("page_url", StringType()),
        StructField("referrer", StringType()),
        StructField("device_type", StringType()),
        StructField("search_term", StringType()),
        StructField("order_id", StringType()),
    ]
)

In [0]:
schema_map = {
    'products': products_struct_column_schema,
    'categories': categories_struct_column_schema,
    'inventory': inventory_struct_column_schema,
    'clickstream': clickstream_struct_column_schema,
}

In [0]:
def file_load(_file_format, _schema):

    if _file_format=="csv":

        source_file_df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", _file_format)
        .schema(_schema)
        .option("header", "true")
        .load(source_path)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
        )

    else:
        source_file_df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", _file_format)
        .schema(_schema)
        .load(source_path)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
        )

    return source_file_df

In [0]:
def write_to_target(source_file_df):

    (source_file_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_loc)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(target_table)
    )

In [0]:
_file_format = FILE_FORMAT_MAP[subfolder]
_schema = schema_map[subfolder]


source_file_df= file_load(_file_format, _schema)

write_to_target(source_file_df)

###Validation after the load

In [0]:
query = f"select * from {target_table}"

print(query)

In [0]:
spark.sql(query).display()